<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Introduction</p>

<div style="background-color: #28282B; border: 4px solid #C4A484; border-radius: 40px; padding: 20px; font-family: 'Roboto'; color: #EADDCA; text-align: left; font-size: 120%;">
    <ul style="list-style-type: square; padding-left: 20px;">
        <li><strong>Competition:</strong> Playground Series Season 6, Episode 4 — Predicting Irrigation Need</li>
        <li style="margin-top: 10px;"><strong>Task:</strong> Multiclass classification — predict <code>Irrigation_Need</code> as one of <code>Low</code>, <code>Medium</code>, or <code>High</code></li>
        <li style="margin-top: 10px;"><strong>Metric:</strong> Balanced Accuracy — accounts for class imbalance (High class is ~3% of data)</li>
        <li style="margin-top: 10px;"><strong>Data:</strong> 630,000 training rows, 270,000 test rows, 19 features (8 categorical, 11 numerical), no missing values</li>
        <li style="margin-top: 10px;"><strong>Approach:</strong> LightGBM multiclass with stratified K-Fold cross-validation, optimising for balanced accuracy</li>
    </ul>
</div>

<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Imports</p>

In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'iframe'

In [3]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

pd.options.display.max_columns = None


<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Configuration</p>

In [4]:
class CFG:

    _k1        = Path('/kaggle/input/competitions/playground-series-s6e4')
    _k2        = Path('/kaggle/input/playground-series-s6e4')
    _local     = Path('/Users/alexchilton/Downloads/playground-series-s6e4')
    data_dir   = _k1 if _k1.exists() else (_k2 if _k2.exists() else _local)
    train_path = data_dir / 'train.csv'
    test_path  = data_dir / 'test.csv'
    subm_path  = data_dir / 'sample_submission.csv'

    _ok        = Path('/kaggle/input/irrigation-water-requirement-prediction-dataset')
    _olocal    = Path('/tmp/orig_irrigation')
    orig_dir   = _ok if _ok.exists() else _olocal
    orig_path  = orig_dir / 'irrigation_prediction.csv'
    orig_row_weight = 0.35

    colorscale = 'Sunset'
    color      = '#EADDCA'

    target     = 'Irrigation_Need'
    label_order = ['Low', 'Medium', 'High']  # ascending need

    cat_cols   = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
                  'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

    num_cols   = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
                  'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
                  'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

    n_splits   = 5
    seed       = 42

    lgb_params = {
        'objective':        'multiclass',
        'num_class':        3,
        'metric':           'multi_logloss',
        'num_iterations':   5000,
        'learning_rate':    0.05,
        'num_leaves':       127,
        'max_depth':        -1,
        'min_child_samples': 50,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'reg_lambda':       1.0,
        'reg_alpha':        0.1,
        'verbose':          -1,
        'seed':             42
    }


<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Data Loading</p>

In [5]:
train = pd.read_csv(CFG.train_path)
test  = pd.read_csv(CFG.test_path)
subm  = pd.read_csv(CFG.subm_path)

try:
    orig = pd.read_csv(CFG.orig_path)
    print(f'Orig shape:  {orig.shape}')
except FileNotFoundError:
    orig = pd.DataFrame(columns=train.columns)
    print('Original dataset not found — orig blending disabled')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
display(train.head())


Train shape: (630000, 21)
Test shape:  (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [6]:
print('Dtypes:')
print(train.dtypes)
print(f'\nMissing values — train: {train.isnull().sum().sum()}, test: {test.isnull().sum().sum()}')

Dtypes:
id                           int64
Soil_Type                   object
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Crop_Type                   object
Crop_Growth_Stage           object
Season                      object
Irrigation_Type             object
Water_Source                object
Field_Area_hectare         float64
Mulching_Used               object
Previous_Irrigation_mm     float64
Region                      object
Irrigation_Need             object
dtype: object

Missing values — train: 0, test: 0


<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Exploratory Data Analysis</p>

In [7]:
class EDA:

    def __init__(self, colorscale, color, df):
        self.colorscale = colorscale
        self.color = color
        self.df = df

    def _template(self, fig, title):
        fig.update_layout(
            title=title,
            title_x=0.5,
            plot_bgcolor='rgba(40, 40, 43, 1)',
            paper_bgcolor='rgba(40, 40, 43, 1)',
            font=dict(color=self.color),
            margin=dict(l=72, r=72, t=72, b=72),
            height=500
        )
        return fig

    def distribution_plot(self, col, title):
        fig = px.histogram(
            self.df, x=col, nbins=80,
            color_discrete_sequence=[self.color]
        )
        fig.update_layout(
            xaxis_title=col, yaxis_title='Count',
            bargap=0.05,
            xaxis=dict(gridcolor='grey'),
            yaxis=dict(gridcolor='grey', zerolinecolor='grey')
        )
        fig.update_traces(hovertemplate='Value: %{x:.2f}<br>Count: %{y:,}')
        self._template(fig, title).show()

    def bar_chart(self, col, title, sort=True):
        vc = self.df[col].value_counts().reset_index()
        vc.columns = [col, 'count']
        if sort:
            vc = vc.sort_values('count', ascending=False)
        fig = px.bar(
            vc, x=col, y='count',
            color='count', color_continuous_scale=self.colorscale
        )
        fig.update_layout(
            xaxis_title=col, yaxis_title='Count', bargap=0.2,
            xaxis=dict(gridcolor='grey'),
            yaxis=dict(gridcolor='grey', zerolinecolor='grey'),
            plot_bgcolor='rgba(40, 40, 43, 1)',
            paper_bgcolor='rgba(40, 40, 43, 1)',
            font=dict(color=self.color),
            margin=dict(l=40, r=40, t=60, b=40),
            height=500
        )
        fig.update_traces(
            hovertemplate='<b>%{x}</b><br>Count: %{y:,}',
            marker=dict(line=dict(width=1, color='DarkSlateGrey'))
        )
        self._template(fig, title).show()

    def stacked_bar(self, col, title):
        """Show class distribution within each category value."""
        ct = self.df.groupby([col, CFG.target]).size().reset_index(name='count')
        totals = ct.groupby(col)['count'].transform('sum')
        ct['pct'] = ct['count'] / totals * 100
        fig = px.bar(
            ct, x=col, y='pct', color=CFG.target,
            color_discrete_sequence=px.colors.sequential.Sunset,
            category_orders={CFG.target: CFG.label_order},
            barmode='stack',
            text_auto='.1f'
        )
        fig.update_layout(
            xaxis_title=col, yaxis_title='% share',
            plot_bgcolor='rgba(40, 40, 43, 1)',
            paper_bgcolor='rgba(40, 40, 43, 1)',
            font=dict(color=self.color),
            margin=dict(l=40, r=40, t=60, b=40),
            height=500
        )
        self._template(fig, title).show()

    def box_by_target(self, col, title):
        """Box plot of a numeric feature split by target class."""
        fig = px.box(
            self.df, x=CFG.target, y=col,
            color=CFG.target,
            color_discrete_sequence=px.colors.sequential.Sunset,
            category_orders={CFG.target: CFG.label_order}
        )
        fig.update_layout(
            plot_bgcolor='rgba(40, 40, 43, 1)',
            paper_bgcolor='rgba(40, 40, 43, 1)',
            font=dict(color=self.color),
            margin=dict(l=40, r=40, t=60, b=40),
            height=500
        )
        self._template(fig, title).show()

In [8]:
eda = EDA(CFG.colorscale, CFG.color, train)

### Target Distribution

The target is heavily imbalanced — `High` is only ~3% of training samples. This is exactly why **balanced accuracy** is used as the metric.

In [9]:
eda.bar_chart(CFG.target, title='Target Distribution: Irrigation_Need', sort=False)

In [10]:
print(train[CFG.target].value_counts())
print()
print(train[CFG.target].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Irrigation_Need
Low       58.72 %
Medium    37.95 %
High       3.33 %
Name: proportion, dtype: object


### Numerical Features

In [11]:
eda.distribution_plot('Soil_pH', 'Soil pH Distribution')
eda.box_by_target('Soil_pH', 'Soil pH by Irrigation Need')

In [12]:
eda.distribution_plot('Soil_Moisture', 'Soil Moisture Distribution')
eda.box_by_target('Soil_Moisture', 'Soil Moisture by Irrigation Need')

In [13]:
eda.distribution_plot('Rainfall_mm', 'Rainfall (mm) Distribution')
eda.box_by_target('Rainfall_mm', 'Rainfall by Irrigation Need')

In [14]:
eda.distribution_plot('Temperature_C', 'Temperature (°C) Distribution')
eda.box_by_target('Temperature_C', 'Temperature by Irrigation Need')

In [15]:
eda.distribution_plot('Previous_Irrigation_mm', 'Previous Irrigation (mm) Distribution')
eda.box_by_target('Previous_Irrigation_mm', 'Previous Irrigation by Irrigation Need')

In [16]:
eda.distribution_plot('Humidity', 'Humidity Distribution')
eda.box_by_target('Humidity', 'Humidity by Irrigation Need')

In [17]:
eda.distribution_plot('Sunlight_Hours', 'Sunlight Hours Distribution')
eda.box_by_target('Sunlight_Hours', 'Sunlight Hours by Irrigation Need')

In [18]:
eda.distribution_plot('Wind_Speed_kmh', 'Wind Speed (km/h) Distribution')
eda.box_by_target('Wind_Speed_kmh', 'Wind Speed by Irrigation Need')

In [19]:
eda.distribution_plot('Field_Area_hectare', 'Field Area (ha) Distribution')
eda.box_by_target('Field_Area_hectare', 'Field Area by Irrigation Need')

### Categorical Features

In [20]:
eda.bar_chart('Soil_Type', 'Soil Type')
eda.stacked_bar('Soil_Type', 'Irrigation Need by Soil Type')

In [21]:
eda.bar_chart('Crop_Type', 'Crop Type')
eda.stacked_bar('Crop_Type', 'Irrigation Need by Crop Type')

In [22]:
eda.bar_chart('Crop_Growth_Stage', 'Crop Growth Stage')
eda.stacked_bar('Crop_Growth_Stage', 'Irrigation Need by Crop Growth Stage')

In [23]:
eda.bar_chart('Season', 'Season')
eda.stacked_bar('Season', 'Irrigation Need by Season')

In [24]:
eda.bar_chart('Irrigation_Type', 'Irrigation Type')
eda.stacked_bar('Irrigation_Type', 'Irrigation Need by Irrigation Type')

In [25]:
eda.bar_chart('Water_Source', 'Water Source')
eda.stacked_bar('Water_Source', 'Irrigation Need by Water Source')

In [26]:
eda.bar_chart('Mulching_Used', 'Mulching Used')
eda.stacked_bar('Mulching_Used', 'Irrigation Need by Mulching')

In [27]:
eda.bar_chart('Region', 'Region')
eda.stacked_bar('Region', 'Irrigation Need by Region')

### Correlation Heatmap (Numerical Features)

In [28]:
corr = train[CFG.num_cols].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale='RdBu',
    zmid=0,
    text=corr.round(2).values,
    texttemplate='%{text}'
))

fig.update_layout(
    title='Correlation Matrix — Numerical Features',
    title_x=0.5,
    plot_bgcolor='rgba(40, 40, 43, 1)',
    paper_bgcolor='rgba(40, 40, 43, 1)',
    font=dict(color='#EADDCA'),
    height=600
)
fig.show()

<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Feature Engineering & Encoding</p>

In [29]:
# Label encode target: Low=0, Medium=1, High=2
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
inv_label_map = {v: k for k, v in label_map.items()}

y = train[CFG.target].map(label_map).values

# Ordinal encode categoricals — fit on train, apply to test + orig
encoders = {}
X_train = train.drop(columns=['id', CFG.target]).copy()
X_test  = test.drop(columns=['id']).copy()

for col in CFG.cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col]  = le.transform(X_test[col])
    encoders[col] = le

# Encode original dataset if available
use_orig = len(orig) > 0 and CFG.target in orig.columns
if use_orig:
    X_orig = orig.drop(columns=[CFG.target]).copy()
    y_orig = orig[CFG.target].map(label_map).values
    for col in CFG.cat_cols:
        X_orig[col] = encoders[col].transform(X_orig[col])
    print(f'X_orig shape: {X_orig.shape}')
    print(f'Orig target dist: {pd.Series(y_orig).value_counts().to_dict()}')
else:
    X_orig, y_orig = None, None
    print('No original data — training on synthetic only')

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
display(X_train.head())


X_train shape: (630000, 19)
X_test shape:  (270000, 19)


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,1,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,4,2,2,1,1,0.82,0,112.16,1
1,0,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,5,3,0,2,3,5.27,1,47.16,3
2,0,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,3,3,0,3,2,8.24,1,110.38,2
3,2,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,5,0,0,0,3,8.32,1,53.85,3
4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,5,2,1,0,3,7.37,0,93.19,3


<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">LightGBM — Stratified K-Fold</p>

In [30]:
skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)

oof_preds  = np.zeros((len(X_train), 3))
test_preds = np.zeros((len(X_test), 3))
fold_scores = []
models = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y)):

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y[train_idx], y[valid_idx]

    # Augment fold training data with original dataset
    if use_orig:
        X_tr_aug = pd.concat([X_tr, X_orig], ignore_index=True)
        y_tr_aug = np.concatenate([y_tr, y_orig])
        sw_syn  = compute_sample_weight('balanced', y_tr)
        sw_orig = np.full(len(y_orig), CFG.orig_row_weight)
        sw      = np.concatenate([sw_syn, sw_orig])
    else:
        X_tr_aug = X_tr
        y_tr_aug = y_tr
        sw = compute_sample_weight('balanced', y_tr)

    model = lgb.LGBMClassifier(**CFG.lgb_params)
    model.fit(
        X_tr_aug, y_tr_aug,
        sample_weight=sw,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(200)
        ]
    )

    val_probs = model.predict_proba(X_val)
    val_preds = val_probs.argmax(axis=1)

    score = balanced_accuracy_score(y_val, val_preds)
    fold_scores.append(score)
    print(f'Fold {fold+1} | Balanced Accuracy: {score:.4f}')

    oof_preds[valid_idx]  = val_probs
    test_preds            += model.predict_proba(X_test) / CFG.n_splits
    models.append(model)

print(f'\nMean CV Balanced Accuracy: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')


[200]	valid_0's multi_logloss: 0.0643779
[400]	valid_0's multi_logloss: 0.0596256
[600]	valid_0's multi_logloss: 0.0590847
Fold 1 | Balanced Accuracy: 0.9644
[200]	valid_0's multi_logloss: 0.0659161
[400]	valid_0's multi_logloss: 0.0610048
[600]	valid_0's multi_logloss: 0.0606057
Fold 2 | Balanced Accuracy: 0.9659
[200]	valid_0's multi_logloss: 0.0645972
[400]	valid_0's multi_logloss: 0.059973
[600]	valid_0's multi_logloss: 0.0596498
Fold 3 | Balanced Accuracy: 0.9666
[200]	valid_0's multi_logloss: 0.0643944
[400]	valid_0's multi_logloss: 0.0594601
[600]	valid_0's multi_logloss: 0.0587756
Fold 4 | Balanced Accuracy: 0.9642
[200]	valid_0's multi_logloss: 0.0630757
[400]	valid_0's multi_logloss: 0.0582012
[600]	valid_0's multi_logloss: 0.0577023
Fold 5 | Balanced Accuracy: 0.9648

Mean CV Balanced Accuracy: 0.9652 ± 0.0009


In [31]:
# Overall OOF balanced accuracy
oof_labels = oof_preds.argmax(axis=1)
oof_score  = balanced_accuracy_score(y, oof_labels)
print(f'OOF Balanced Accuracy (full train): {oof_score:.4f}')

# CV score plot
fold_scores_r = [round(s, 4) for s in fold_scores]
mean_score    = round(np.mean(fold_scores_r), 4)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, CFG.n_splits + 1)), y=fold_scores_r,
    mode='markers',
    name='Fold Scores',
    marker=dict(size=18, color='#EADDCA', symbol='diamond'),
    text=[f'{s:.4f}' for s in fold_scores_r],
    hovertemplate='Fold %{x}: %{text}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=[1, CFG.n_splits], y=[mean_score, mean_score],
    mode='lines', name=f'Mean: {mean_score:.4f}',
    line=dict(dash='dash', color='#FFBF00'),
    hoverinfo='none'
))
fig.update_layout(
    title=f'LightGBM | CV Balanced Accuracy: {mean_score:.4f}',
    title_x=0.5, xaxis_title='Fold', yaxis_title='Balanced Accuracy',
    plot_bgcolor='rgba(40, 40, 43, 1)', paper_bgcolor='rgba(40, 40, 43, 1)',
    font=dict(color='#EADDCA'),
    xaxis=dict(gridcolor='grey', tickmode='linear', tick0=1, dtick=1,
               range=[0.5, CFG.n_splits + 0.5], zerolinecolor='grey'),
    yaxis=dict(gridcolor='grey', zerolinecolor='grey'),
    height=480
)
fig.show()

OOF Balanced Accuracy (full train): 0.9652


### Feature Importance

In [32]:
# Average feature importance across folds
fi = pd.DataFrame({
    'feature': X_train.columns,
    'importance': np.mean([m.feature_importances_ for m in models], axis=0)
}).sort_values('importance', ascending=False)

fig = px.bar(
    fi, x='importance', y='feature', orientation='h',
    color='importance', color_continuous_scale='Sunset'
)
fig.update_layout(
    title='Mean Feature Importance (gain)',
    title_x=0.5,
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='rgba(40, 40, 43, 1)',
    paper_bgcolor='rgba(40, 40, 43, 1)',
    font=dict(color='#EADDCA'),
    height=600
)
fig.show()

<p style="background-color: #EADDCA; font-size: 300%; text-align: center; border-radius: 40px 40px; color: #C4A484; font-weight: bold; font-family: 'Roboto'; border: 4px solid #C4A484; text-shadow: 2px 2px 0.5px rgba(0, 0, 0, 0.05);">Submission</p>

In [33]:
test_labels = test_preds.argmax(axis=1)
test_classes = [inv_label_map[l] for l in test_labels]

subm[CFG.target] = test_classes
subm.to_csv('submission.csv', index=False)

print('Submission saved.')
print(f'\nTest prediction distribution:')
print(pd.Series(test_classes).value_counts())
display(subm.head(10))

Submission saved.

Test prediction distribution:
Low       159767
Medium    101378
High        8855
Name: count, dtype: int64


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
5,630005,Medium
6,630006,Low
7,630007,Medium
8,630008,Medium
9,630009,Low
